# 🎙️ Gemma 4 · 會議音檔轉錄 Demo

**流程：** GPU 確認 → 安裝依賴 → Clone 專案 → 啟動模型服務 → 啟動後端 → 取得 Demo URL

> 每個 Cell 獨立執行，若某步驟失敗請單獨重跑該 Cell。

## Cell 0 — GPU 確認

In [ ]:
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('❌ 未偵測到 GPU。請至 Runtime → Change runtime type → GPU (T4 / A100)')
    sys.exit(1)
else:
    print(result.stdout)
    print('✅ GPU 就緒')

## Cell 1 — 安裝系統依賴 (ffmpeg)

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!ffmpeg -version 2>&1 | head -1
print('✅ ffmpeg 安裝完成')

## Cell 2 — Clone 專案

In [ ]:
import os

REPO_URL = 'https://github.com/simonliu-ai-product/Gemma-Meeting-Notes.git'
PROJECT_DIR = '/content/Gemma-Meeting-Notes'

if os.path.exists(PROJECT_DIR):
    print('📁 目錄已存在，執行 git pull ...')
    !git -C {PROJECT_DIR} pull
else:
    print('📥 Clone 中 ...')
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f'✅ 工作目錄：{os.getcwd()}')
!ls -la

## Cell 3 — 安裝 Python 依賴

In [ ]:
!pip install -q -U "transformers>=5.5.0"
!pip install -q -r backend/requirements.txt
import transformers
print(f'✅ Python 依賴安裝完成 (transformers {transformers.__version__})')

## Cell 4 — HuggingFace 登入（選填）

> Gemma 4 採用 Apache 2.0 授權，**不需要**事先同意 gated model 條款。
> 若有 HuggingFace token 可登入以提升下載速度，沒有也可直接跳到 Cell 5。

In [ ]:
# 選填：有 token 可加速下載，沒有直接跳到 Cell 5
# 取得 Token：https://huggingface.co/settings/tokens
from huggingface_hub import login
from getpass import getpass

hf_token = getpass('HuggingFace Token（直接按 Enter 跳過）：')
if hf_token.strip():
    login(token=hf_token, add_to_git_credential=False)
    print('✅ HuggingFace 登入成功')
else:
    print('⏭️  跳過登入，直接使用公開下載')

## Cell 5 — 啟動模型服務（port 8001）

> ⏳ 首次執行需下載模型（E4B ≈ 9 GB），請耐心等候。
> 看到 `✅ 模型服務就緒` 才繼續下一步。

In [ ]:
import subprocess, time, httpx, os

env = os.environ.copy()
env['MODEL_SERVICE_PORT'] = '8001'
# 切換成 E2B 只需改這一行（注意大寫 E）
env['GEMMA_MODEL_ID'] = 'google/gemma-4-E4B-it'

model_proc = subprocess.Popen(
    ['python', 'backend/model_service.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print('⏳ 模型載入中（首次需要數分鐘）...')
for _ in range(300):  # 最多等 5 分鐘
    time.sleep(2)
    try:
        r = httpx.get('http://localhost:8001/health', timeout=3)
        if r.status_code == 200:
            info = r.json()
            print(f'\n✅ 模型服務就緒')
            print(f'   Model  : {info["model"]}')
            print(f'   Device : {info["device"]}')
            break
    except Exception:
        pass
    import select
    if select.select([model_proc.stdout], [], [], 0)[0]:
        line = model_proc.stdout.readline()
        if line:
            print(line, end='')
else:
    print('❌ 模型服務啟動超時，請查看上方 log')

## Cell 6 — 啟動後端 API 服務（port 8000）

In [ ]:
import subprocess, time, httpx, sys, os, select, queue, threading

env = os.environ.copy()
env['MODEL_SERVICE_URL'] = 'http://localhost:8001'

backend_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app',
     '--host', '0.0.0.0', '--port', '8000', '--app-dir', 'backend'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# 背景執行緒持續讀 log，避免 stdout buffer 塞滿
log_lines = []
def _reader(proc, lines):
    for line in proc.stdout:
        lines.append(line)
        print(line, end='', flush=True)

t = threading.Thread(target=_reader, args=(backend_proc, log_lines), daemon=True)
t.start()

print('⏳ 後端啟動中...')
for _ in range(30):
    time.sleep(1)
    if backend_proc.poll() is not None:
        print(f'\n❌ 後端程序已退出 (return code: {backend_proc.returncode})')
        break
    try:
        r = httpx.get('http://localhost:8000/api/health', timeout=3)
        if r.status_code == 200:
            print('\n✅ 後端服務就緒 → http://localhost:8000')
            break
    except Exception:
        pass
else:
    print('\n❌ 後端啟動超時')

## Cell 7 — 開啟 Demo 視窗

> 執行後 Colab 會自動彈出新分頁，直接開啟前端介面。

In [ ]:
from google.colab import output

print('🚀 開啟 Demo 視窗...')
output.serve_kernel_port_as_window(8000)

## Cell 8 — 停止所有服務（Demo 結束後執行）

In [ ]:
try:
    model_proc.terminate()
    print('✅ 模型服務已停止')
except NameError:
    pass

try:
    backend_proc.terminate()
    print('✅ 後端服務已停止')
except NameError:
    pass

print('✅ 所有服務已停止')